# Gemini Embedding Models (FREE)


In [ ]:
# !pip install google-genai==2.24.0 scikit-learn==1.9.1 python-dotenv

In [13]:
# check package varsion
import importlib.metadata

# Use the distribution names (what you use in pip install)
packages = ["google-genai", "scikit-learn"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


google-genai version: 2.25.0
scikit-learn version: 1.9.0


In [2]:
import getpass
import os
from dotenv import load_dotenv
from google import genai
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load environment variables from .env file
load_dotenv()

# Get API key from environment, or prompt user if not found
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
  api_key = getpass.getpass("Enter your Gemini API key: ")

# Create Gemini client
client = genai.Client(api_key=api_key)


Enter your Gemini API key:  ········


In [8]:
model_emb = "gemini-embedding-001"

In [4]:
# Sentence to embed
sentence = "Machine learning is a subset of artificial intelligence."

# Generate embedding
result = client.models.embed_content(
    model=model_emb,
    contents=sentence
)

# Extract embedding vector
vector = result.embeddings[0].values

print("Sentence:", sentence)

print("\nEmbedding (1st 20 numbers):")
print(vector[:20])

Sentence: Machine learning is a subset of artificial intelligence.

Embedding (1st 20 numbers):
[-0.026853403, -0.0049841655, -0.016345479, -0.046370346, -0.02222475, -0.008440223, 0.00544005, 0.0032457092, 0.0038371943, 0.0025398768, 0.0038405573, -0.004622712, -0.020179315, 0.035211112, 0.13777858, 0.00038011387, -0.009986013, -0.011922514, 0.0035229574, -0.023845322]


In [5]:
print("\nEmbedding dimension:", len(vector))


Embedding dimension: 3072


In [7]:
# 2. Define our documents (text to convert into vectors)

documents = [
    "How to reset your account password.",
    "The weather is very sunny and clear today.",
    "Updating user login credentials and security phrases."
]

# 3. Convert the documents into vector embeddings
result = client.models.embed_content(
    model=model_emb,
    contents=documents
)

# Extract vectors
doc_vectors = [embedding.values for embedding in result.embeddings]

print("--- Vector Preview ---")
print("Document 1 vector shape:", len(doc_vectors[0]))
print("First 20 numbers of Document 1 vector:\n", doc_vectors[0][:20])

--- Vector Preview ---
Document 1 vector shape: 3072
First 20 numbers of Document 1 vector:
 [0.037980016, 0.041285515, 0.0177221, -0.03703116, -0.01364855, 0.01431278, 0.023503676, 0.004174536, -0.010560614, 0.007331381, -0.015484137, -0.012669007, -0.01408812, 0.021638522, 0.11909661, -0.015612938, -0.002107154, -0.0013967425, 0.0022920887, -0.020149665]


In [10]:
# Process a user search query

query = "I forgot how to log into my profile"

# Convert query into a Gemini embedding
query_result = client.models.embed_content(
    model=model_emb,
    contents=query,
)

# Extract the query vector
query_vector = query_result.embeddings[0].values

# Convert to 2D NumPy array because cosine_similarity expects 2D input
query_vector = np.array(query_vector).reshape(1, -1)

# 5. Calculate similarity
# This measures the mathematical angle between the vectors
similarities = cosine_similarity(query_vector, doc_vectors)[0]

print("--- Search Results (Semantic Match) ---")

for i, score in enumerate(similarities):
    print(f"Score: {score:.4f} | Document: '{documents[i]}'")

--- Search Results (Semantic Match) ---
Score: 0.6864 | Document: 'How to reset your account password.'
Score: 0.4885 | Document: 'The weather is very sunny and clear today.'
Score: 0.6379 | Document: 'Updating user login credentials and security phrases.'


Document 1 and 3 are similar to the query